# Practice 30 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — Debug

### Task 1: find and fix 4 bugs

Each snippet below has one bug. Identify what's wrong and write the corrected version.

**Snippet A**
```python
titanic = sns.load_dataset('titanic')
before = titanic.memory_usage().sum()
titanic[['sex', 'embarked']] = titanic[['sex', 'embarked']].astype('category')
after = titanic.memory_usage().sum()
print(f"Saved: {before - after} bytes")
```

**Snippet B**
```python
diamonds = sns.load_dataset('diamonds')
diamonds['cut'] = diamonds['cut'].astype('category')
cheap = diamonds[diamonds['price'] < 500]
cheap['cut'].value_counts()  # should only show cuts that actually appear
```

**Snippet C**
```python
tips = sns.load_dataset('tips')
tips['day'] = tips['day'].astype('category')
result = tips.groupby('day')['tip'].mean()
```

**Snippet D**
```python
tips = sns.load_dataset('tips')
tips.set_index('day')
print(tips.loc['Fri'])
```

In [3]:
# Your fixes here
#A
titanic = sns.load_dataset('titanic')
before = titanic.memory_usage(deep=True).sum()
titanic[['sex', 'embarked']] = titanic[['sex', 'embarked']].astype('category')
after = titanic.memory_usage(deep=True).sum()
print(f"Saved: {before - after} bytes")

Saved: 104309 bytes


In [7]:
#B
diamonds = sns.load_dataset('diamonds')
diamonds['cut'] = diamonds['cut'].astype('category')
cheap = diamonds[diamonds['price'] < 500].copy()
cheap['cut'] = cheap['cut'].cat.remove_unused_categories()
cheap['cut'].value_counts()  # should only show cuts that actually appear

cut
Very Good    653
Ideal        628
Good         226
Premium      215
Fair           7
Name: count, dtype: int64

In [11]:
#C
tips = sns.load_dataset('tips')
tips['day'] = tips['day'].astype('category')
result = tips.groupby('day', observed=True)['tip'].mean()

In [15]:
#D
tips = sns.load_dataset('tips')
tips = tips.set_index('day')
print(tips.loc['Fri'])

     total_bill   tip     sex smoker    time  size
day                                               
Fri       28.97  3.00    Male    Yes  Dinner     2
Fri       22.49  3.50    Male     No  Dinner     2
Fri        5.75  1.00  Female    Yes  Dinner     2
Fri       16.32  4.30  Female    Yes  Dinner     2
Fri       22.75  3.25  Female     No  Dinner     2
Fri       40.17  4.73    Male    Yes  Dinner     4
Fri       27.28  4.00    Male    Yes  Dinner     2
Fri       12.03  1.50    Male    Yes  Dinner     2
Fri       21.01  3.00    Male    Yes  Dinner     2
Fri       12.46  1.50    Male     No  Dinner     2
Fri       11.35  2.50  Female    Yes  Dinner     2
Fri       15.38  3.00  Female    Yes  Dinner     2
Fri       12.16  2.20    Male    Yes   Lunch     2
Fri       13.42  3.48  Female    Yes   Lunch     2
Fri        8.58  1.92    Male    Yes   Lunch     1
Fri       15.98  3.00  Female     No   Lunch     3
Fri       13.42  1.58    Male    Yes   Lunch     2
Fri       16.27  2.50  Female  

---
## Level 2 — Exploration

### Task 2: planets — new dataset, open questions

Load `sns.load_dataset('planets')`. This dataset records exoplanet discoveries: columns include `method` (detection method), `number` (planets discovered), `orbital_period`, `mass`, `distance`, and `year`.

No structure — just answer these questions:

1. Check the dtypes. Which columns look like good candidates for `category` conversion? Convert them and compare memory before and after (`deep=True`).
2. How many unique detection methods are there? Print the `.cat.codes` for the first 5 rows of `method`.
3. Which method has discovered the most planets in total? (Hint: `number` counts planets per row, not just 1 per row.)
4. What is the median `orbital_period` per detection method? Which method has the longest? Use `observed=True` and NumPy.

In [36]:
# Your code here
planets = sns.load_dataset('planets')
planets.dtypes
before = planets.memory_usage(deep = True)
planets['method'] = planets['method'].astype('category')
after = planets.memory_usage(deep = True)
before.sum()-after.sum()


planets['method'].cat.codes[0:5]
len(planets['method'].cat.categories)

planets.groupby('method',observed=True)['number'].sum().idxmax()

m =planets.groupby('method',observed=True)['orbital_period'].median()

m

m.index[np.argmax(m)]

'Imaging'

---
## Level 3 — Reshape Challenge

### Task 3: titanic — reshape without pipe

Load `sns.load_dataset('titanic')`. No `.pipe()` in this task.

Using an ordered `CategoricalDtype` for `pclass` (`3 < 2 < 1`) and `category` for `sex`:

- Build a pivot table: **survival rate** (`survived`) by `pclass` × `sex`, with `observed=True`.
- Stack it into a Series. Set the MultiIndex level names to `['pclass', 'sex']` if they aren't already.
- Use `.xs()` with a tuple to extract the survival rate for first-class females in one call.
- Now unstack `sex` back out — you should recover the original pivot table shape.
- Use `.loc` to filter the stacked Series to only passengers in the top two classes (use the ordered comparison, not `.isin()`).

In [ ]:
# Your code here
titanic = sns.load_dataset('titanic')
titanic['sex'] = titanic['sex'].astype('category')
po= pd.CategoricalDtype(categories=[3,2,1],ordered=True)
titanic['pclass'] = titanic['pclass'].astype(po)

p = titanic.pivot_table(
    index = 'pclass',
    columns= 'sex',
    values='survived',
    observed=True
)
p

s = p.stack()
s.xs((1,'female'))

s.unstack('sex')

###don't know how to do the last task
s.loc[titanic['pclass'][titanic['pclass']>3].unique()]


pclass  sex   
1       female    0.968085
        male      0.368852
2       female    0.921053
        male      0.157407
dtype: float64

---
## Level 4 — Pipe (one time)

### Task 4: diamonds — full pipeline

Load `sns.load_dataset('diamonds')`. Write a `.pipe()` chain with exactly 3 functions:

- One applies ordered `CategoricalDtype` to `cut` and `color`
- One adds `price_per_carat` and a cut-level average `price_per_carat` using `transform`
- One filters to diamonds with cut **strictly better than** `'Good'` — use the ordered comparison

After the chain:
- Which color has the highest mean `price_per_carat` among the filtered diamonds? Use `np.argmax()`.
- Build a pivot table of mean `price_per_carat` by `cut` × `color` (`observed=True`). Use a list comprehension to find any `(cut, color)` combinations with no data.

In [85]:
# Your code here

diamonds = sns.load_dataset('diamonds')

def to_cat(df):
    cuto = pd.CategoricalDtype(categories=['Fair','Good','Very Good','Premium','Ideal'],
                         ordered=True)
    df['cut'] = df['cut'].astype(cuto)
    coloro = pd.CategoricalDtype(categories=['D','E','F','G','H','I','J'],
                             ordered=True)
    df['color'] = df['color'].astype(coloro)
    return df

def add_price_per_carat(df):
    df['ppc'] = df['price']/df['carat']
    df['cppc'] = df.groupby('cut', observed = True)['ppc'].transform('mean')
    return df
def fil(df):
    fdf = df[df['cut']>'Good'].copy()
    return fdf 

res = (
    diamonds.copy()
    .pipe(to_cat)
    .pipe(add_price_per_carat)
    .pipe(fil)
)

m = res.groupby('color',observed=True)['ppc'].mean()
m.index[np.argmax(m)]

p = res.pivot_table(
    index = 'cut',
    columns = 'color',
    values = 'ppc',
    observed = True
)

[mi for mi in p.stack().index if pd.isna(p.loc[mi[0], mi[1]])]

[]